# Kiểm chứng toạ độ UAV và Đo lường độ trôi sai số tích luỹ (Accumulated Drift Evaluation)

### 1. Hiện tượng trôi sai số (Drift) là gì và vì sao nguy hiểm?
Khi ghép 2 bức ảnh liên tiếp, sai số lệch 1 hoặc 2 pixel là rất nhỏ và mắt thường không nhận ra. Tuy nhiên, khi máy bay bay xa và ghép liên tục 20, 50 hay 100 bức ảnh:
- Sai số của bức ảnh thứ nhất sẽ cộng dồn vào bức ảnh thứ hai, rồi tiếp tục cộng dồn vào các bức ảnh sau.
- Hiện tượng sai số tích tụ này gọi là **Drift (Độ trôi tích luỹ)**.
- Hậu quả: Toàn bộ bức ảnh bản đồ lớn sẽ bị bẻ cong, vặn xoắn hoặc méo mó hình học nghiêm trọng so với mặt đất thực tế.

### 2. Cấu trúc và Mục tiêu của notebook này
Notebook gồm 2 phần được phân tách rõ ràng:

1. **Phần 1 (Mục 1-7: Khảo sát chẩn đoán toạ độ Drone trong `trajectory.txt`):**
   - Chẩn đoán tương thích hình học giữa toạ độ pose và homography ảnh.
   - Phát hiện giới hạn: pose do SLAM tái dựng, tỷ lệ mét kế thừa giả định mặt phẳng phẳng, dữ liệu grass bị suy biến.
   - Phần này được thiết kế độc lập, không chặn quy trình đo drift chính nếu metadata pose bị thiếu hoặc camera không phải PinHole.

2. **Phần 2 (Mục 8-10: Thước đo độc lập Chain vs Direct theo Quyết định `RDR-0007`):**
   - Đo sự bất đồng tương đối (độ trôi tự nhất quán - self-consistency) giữa chuỗi ghép tuần tự từ benchmark và phép ghép trực tiếp bằng SIFT ở ảnh gốc.
   - Áp dụng gate nghiêm ngặt theo RDR-0007 (chỉ nhận điểm đo khi cặp ghép trực tiếp có ít nhất 100 inlier; khi bay xa mất vùng chồng ghi nhận không khả dụng NA trung thực, không giả lập ground-truth).
   - So sánh giữa các mô hình dựa trên tập khung hình đối chiếu chung (common eligible frames) để đảm bảo tính công bằng.


In [ ]:
# ==============================================================================
# CẤU HÌNH NHANH (QUICK CONFIG)
# - UAV_INPUT_RUN: None để tự động lấy run benchmark mới nhất, hoặc gán đường dẫn run cụ thể.
# - ENABLE_POSE_DIAGNOSIS: False (mặc định đo drift độc lập) hoặc True.
# ==============================================================================
UAV_INPUT_RUN = None  # None hoặc đường dẫn tới thư mục run (VD: "outputs/03_all_models/...")
ENABLE_POSE_DIAGNOSIS = False  # False hoặc True

import os
if UAV_INPUT_RUN:
    os.environ["UAV_INPUT_RUN"] = str(UAV_INPUT_RUN)
if ENABLE_POSE_DIAGNOSIS:
    os.environ["UAV_ENABLE_POSE_DIAGNOSIS"] = "1"

import hashlib
import itertools
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

# 1. Cloud environment guard (RULES.md §8: Experiments must run on Google Colab or Kaggle hosted runtime only)
is_colab = bool(os.environ.get("COLAB_RELEASE_TAG")) and Path("/content").is_dir()
is_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) and Path("/kaggle").is_dir()
if not (is_colab or is_kaggle):
    raise RuntimeError(
        "Unsupported runtime: Execution is restricted to hosted Google Colab or Kaggle runtimes (RULES.md §8). "
        "Local execution is disabled to prevent unverified local runs."
    )

# 2. Mount Google Drive if on Colab - fail closed if mount fails
if is_colab:
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.ismount("/content/drive") or not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive mount failed; persistent storage unavailable.")

def resolve_paths():
    """Một root dữ liệu rõ ràng; output Colab bắt buộc nằm trên Drive đã mount."""
    default_project = "/content/drive/MyDrive/mosaicking_from_uav_video_research" if is_colab else "/kaggle/working"
    project_dir = Path(os.environ.get("UAV_PROJECT_ROOT", default_project)).expanduser().resolve()
    if not project_dir.is_dir():
        raise FileNotFoundError(f"Project directory missing: {project_dir}")
    data_dir = next((project_dir / name for name in ("dataset", "data")
                     if (project_dir / name).is_dir()), None)
    if data_dir is None:
        raise FileNotFoundError(f"Expected dataset/ or data/ inside {project_dir}")
    if is_kaggle and not os.environ.get("UAV_OUTPUT_ROOT"):
        raise ValueError("Set UAV_OUTPUT_ROOT and save Kaggle version/output for persistent handoff.")
    default_output = "/content/drive/MyDrive/mosaicking_from_uav_video_research/outputs"
    output_root = Path(os.environ.get("UAV_OUTPUT_ROOT", default_output)).expanduser().resolve()
    if is_colab and not output_root.is_relative_to(Path("/content/drive").resolve()):
        raise ValueError(f"Colab output must be on mounted Google Drive: {output_root}")
    return project_dir, data_dir, output_root

PROJECT_DIR, DATA_DIR, OUTPUT_ROOT = resolve_paths()
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")

# Tạo thư mục chạy độc bản UTC theo format OUTPUT_ROOT/04_drift_evaluation/<stamp>
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = OUTPUT_ROOT / "04_drift_evaluation" / STAMP
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
print(f"RUN_DIR khởi tạo thành công: {RUN_DIR}")

def verify_run_structural_completeness(run_p: Path) -> tuple:
    """Kiểm tra tính toàn vẹn cấu trúc của một run benchmark."""
    manifest_p = run_p / "manifest.json"
    if not manifest_p.is_file():
        return False, {}
    try:
        manifest = json.loads(manifest_p.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return False, {}
    seqs = manifest.get("sequences", [])
    methods = manifest.get("methods", [])
    if not seqs or not methods:
        return False, {}
    mosaic_dir = run_p / "mosaic"
    if not mosaic_dir.is_dir():
        return False, {}
    for s in seqs:
        s_dir = mosaic_dir / s
        if not s_dir.is_dir():
            return False, {}
        for m in methods:
            m_metrics = s_dir / m / "metrics.json"
            if not m_metrics.is_file():
                return False, {}
            try:
                data = json.loads(m_metrics.read_text(encoding="utf-8"))
                if "H_to_mosaic" not in data or "frames_stitched" not in data:
                    return False, {}
            except (OSError, ValueError):
                return False, {}
    return True, manifest

def latest_run_dir():
    """
    Xác định run benchmark 03_all_models:
    - Nếu UAV_INPUT_RUN được đặt: cho phép run lịch sử NẾU đã kiểm chứng toàn vẹn cấu trúc mọi sequence/method.
    - Tìm kiếm tự động: CHỈ chấp nhận run có marker completed.json. Không tự động nhận thư mục chưa hoàn tất.
    - Sắp xếp theo tên timestamp (p.name).
    """
    custom = os.environ.get("UAV_INPUT_RUN")
    if custom:
        cand = Path(custom)
        if not cand.is_dir():
            raise FileNotFoundError(f"UAV_INPUT_RUN chỉ định thư mục không tồn tại: {custom}")
        ok, manifest = verify_run_structural_completeness(cand)
        if ok:
            print(f"Sử dụng run lịch sử được chỉ định qua UAV_INPUT_RUN (đã xác thực toàn vẹn cấu trúc): {cand.name}")
            return cand, manifest
        raise ValueError(f"Run chỉ định qua UAV_INPUT_RUN={custom} không hoàn chỉnh cấu trúc và thiếu completed.json.")

    search_dirs = [OUTPUT_ROOT / "03_all_models"]

    completed_candidates = []
    for base in search_dirs:
        if not base.is_dir():
            continue
        for p in base.iterdir():
            if p.is_dir() and (p / "completed.json").is_file() and (p / "manifest.json").is_file():
                completed_candidates.append(p)

    if not completed_candidates:
        raise FileNotFoundError(
            "Không tìm thấy run hoàn tất của feature matcher benchmark (yêu cầu completed.json và manifest.json). "
            "Hãy chạy benchmark trước hoặc chỉ định UAV_INPUT_RUN cho run lịch sử đã xác thực."
        )

    best_candidate = sorted(completed_candidates, key=lambda p: p.name)[-1]
    complete, manifest = verify_run_structural_completeness(best_candidate)
    if not complete:
        raise ValueError(f"Completed marker exists but benchmark artifacts are incomplete: {best_candidate}")
    print(f"Tự động chọn run hoàn tất mới nhất: {best_candidate.name}")
    return best_candidate, manifest

run_dir, benchmark_manifest = latest_run_dir()
manifest_hash = hashlib.sha256((run_dir / "manifest.json").read_bytes()).hexdigest()

# Kế thừa danh sách sequence và model từ chính manifest của benchmark run đã chọn (Full Sequence Support)
SEQUENCES = tuple(benchmark_manifest.get("sequences", []))
METHODS = tuple(benchmark_manifest.get("methods", []))
if not SEQUENCES or not METHODS:
    raise ValueError("Benchmark manifest must declare sequences and methods.")
for names in (SEQUENCES, METHODS):
    if len(set(names)) != len(names) or any(not isinstance(n, str) or n in (".", "..") or "/" in n or "\\" in n for n in names):
        raise ValueError("Manifest sequence/method names must be unique directory names.")
print(f"Danh sách Sequences từ benchmark ({len(SEQUENCES)}): {SEQUENCES}")
print(f"Danh sách Methods từ benchmark ({len(METHODS)}): {METHODS}")

def ensure_sequence(data_dir: Path, seq_name: str) -> Path:
    """
    Đảm bảo thư mục sequence có dữ liệu; hỗ trợ giải nén an toàn và xử lý cấu trúc thư mục lồng nhau.
    Kiểm tra path traversal trên mọi thành viên; không dùng fallback không an toàn.
    """
    target = data_dir / seq_name

    def find_sequence_root(base: Path) -> Path:
        if (base / "rgb").is_dir() and any((base / "rgb").glob("*.jpg")):
            return base
        for rgb_dir in base.rglob("rgb"):
            if rgb_dir.is_dir() and any(rgb_dir.glob("*.jpg")):
                return rgb_dir.parent
        return base

    resolved = find_sequence_root(target)
    if (resolved / "rgb").is_dir() and any((resolved / "rgb").glob("*.jpg")):
        return resolved

    import tarfile, zipfile
    for ext in (".tar.gz", ".tgz", ".zip"):
        archive = data_dir / f"{seq_name}{ext}"
        if archive.is_file():
            print(f"Giải nén an toàn {archive.name} vào {target}...")
            target.mkdir(parents=True, exist_ok=True)
            target_resolved = target.resolve()
            if ext == ".zip":
                with zipfile.ZipFile(archive, "r") as z:
                    for member in z.infolist():
                        dest = (target / member.filename).resolve()
                        if not dest.is_relative_to(target_resolved):
                            raise ValueError(f"Zip path traversal detected: {member.filename}")
                    z.extractall(target)
            else:
                with tarfile.open(archive, "r:*") as t:
                    if not hasattr(tarfile, "data_filter"):
                        raise RuntimeError("Safe tar extraction requires Python with tarfile.data_filter.")
                    t.extractall(target, filter="data")
            break

    resolved = find_sequence_root(target)
    if not ((resolved / "rgb").is_dir() and any((resolved / "rgb").glob("*.jpg"))):
        raise FileNotFoundError(f"Không tìm thấy thư mục ảnh rgb sau khi kiểm tra/giải nén {seq_name}")
    return resolved

seq_roots = {s: ensure_sequence(DATA_DIR, s) for s in SEQUENCES}

# Khám phá danh sách ảnh và kích thước ảnh trực tiếp từ file ảnh (KHÔNG phụ thuộc config.cfg/pose)
images_data = {}
for sequence in SEQUENCES:
    sroot = seq_roots[sequence]
    rgb_dir = sroot / "rgb"
    files = sorted(rgb_dir.glob("*.jpg"), key=lambda p: float(p.stem))
    if not files:
        raise FileNotFoundError(f"Không có ảnh jpg trong {rgb_dir}")
    first_img = cv2.imread(str(files[0]))
    if first_img is None:
        raise ValueError(f"Không thể đọc ảnh {files[0]}")
    h_orig, w_orig = first_img.shape[:2]
    del first_img
    images_data[sequence] = {
        "root": sroot,
        "rgb_dir": rgb_dir,
        "files": files,
        "size": (int(w_orig), int(h_orig)),  # (width, height)
    }
    print(f"{sequence}: {len(files)} ảnh | kích thước gốc: {w_orig}x{h_orig}")

# Cấu hình chẩn đoán pose: MẶC ĐỊNH TẮT (ENABLE_POSE_DIAGNOSIS=0) để tránh lỗi dữ liệu giả và không chặn drift chính
ENABLE_POSE_DIAGNOSIS = os.environ.get("UAV_ENABLE_POSE_DIAGNOSIS", "0") == "1"
if ENABLE_POSE_DIAGNOSIS:
    print("Chẩn đoán pose được BẬT qua UAV_ENABLE_POSE_DIAGNOSIS=1.")
else:
    print("Chẩn đoán pose MẶC ĐỊNH TẮT (UAV_ENABLE_POSE_DIAGNOSIS=0). Đo drift độc lập được ưu tiên.")

def read_config(path):
    """Đọc config.cfg: trả về (K, size, plane, status)."""
    values = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, val = line.split("=", 1)
            values[key.strip()] = val.strip()
    cam_type = values.get("Camera.CameraType", "Unknown")
    if cam_type != "PinHole":
        return None, None, None, f"Camera type '{cam_type}' != 'PinHole'"
    w, h, fx, fy, cx, cy = [float(v) for v in values["Camera.Paraments"].strip("[]").split(",")]
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    plane = [float(v) for v in values["Plane"].split()]
    return K, (int(w), int(h)), plane, "ok"

def read_trajectory(path):
    """Dòng trajectory.txt: timestamp tx ty tz qx qy qz qw (định dạng SE3 Map2DFusion)."""
    rows = [line.split() for line in Path(path).read_text(encoding="utf-8").splitlines() if line.split()]
    return np.asarray([[float(v) for v in r[:8]] for r in rows], dtype=np.float64)

pose_data = {}
if ENABLE_POSE_DIAGNOSIS:
    for sequence in SEQUENCES:
        sroot = images_data[sequence]["root"]
        cfg_candidates = list(sroot.rglob("config.cfg"))
        traj_candidates = list(sroot.rglob("trajectory.txt"))
        if not cfg_candidates or not traj_candidates:
            print(f"Notice: {sequence} thiếu config.cfg hoặc trajectory.txt -> bỏ qua pose.")
            continue
        K, cfg_size, plane, status = read_config(cfg_candidates[0])
        if status != "ok":
            print(f"Notice: {sequence} config không hỗ trợ ({status}) -> bỏ qua pose.")
            continue
        traj = read_trajectory(traj_candidates[0])
        files = images_data[sequence]["files"]
        if len(traj) != len(files):
            print(f"Notice: {sequence}: {len(traj)} pose vs {len(files)} ảnh -> bỏ qua pose.")
            continue
        pose_data[sequence] = dict(
            K=K, size=cfg_size, plane=plane, traj=traj, files=files,
            cfg_path=cfg_candidates[0], traj_path=traj_candidates[0]
        )
        print(f"{sequence} (pose khả dụng): {len(files)} frame | fx={K[0,0]:.0f}")

### 2. Công thức toán học chuyển đổi từ Toạ độ Drone sang Ma trận Homography
Về mặt lý thuyết, nếu biết độ cao bay, góc nghiêng của camera và toạ độ mặt phẳng đất, ta có thể tính ra ma trận biến đổi phối cảnh giữa 2 bức ảnh dựa trên công thức hình học xạ ảnh của camera lỗ kim (PinHole Camera Model):

$$H_{j \to i} = K \left( R_{wc,i} R_{wc,j}^{\top} + \frac{\big(R_{wc,i}(C_j - C_i)\big)\,(R_{wc,j}\,n_w)^{\top}}{d_j} \right) K^{-1}$$

với $d_j = n_w \cdot (p_w - C_j)$ là khoảng cách từ camera $j$ tới mặt phẳng đất.


In [ ]:
def quat_to_R(q):
    """
    Quaternion (qx, qy, qz, qw) -> ma trận xoay 3x3.
    Thứ tự này lấy từ mã nguồn tác giả dataset:
    - Map2DFusion `src/main.cpp` đọc `trajectory.txt` thành `pi::SE3d`.
    - PIL thư viện hình học (`PIL/src/base/types/SE3.h`) định nghĩa `operator>>` là:
      `is >> x >> y >> z >> rx >> ry >> rz >> w` (với rx, ry, rz là vector phần ảo, w là phần thực scalar).
    """
    x, y, z, w = q / np.linalg.norm(q)
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]
    ], dtype=np.float64)

def plane_normal(plane):
    return quat_to_R(np.array(plane[3:8], dtype=np.float64)) @ np.array([0.0, 0.0, 1.0])

def pose(sequence, index, quat_mode="qcw", translation_mode="C=t", use_rotation=True):
    """Trả về (R_wc, C) theo quy ước đang xét cho sequence nếu pose khả dụng."""
    if sequence not in pose_data:
        raise ValueError(f"Không có pose data cho {sequence}")
    row = pose_data[sequence]["traj"][index]
    R = quat_to_R(row[4:8].copy())
    R_cw = R if quat_mode == "qcw" else R.T
    R_wc = R_cw.T if use_rotation else np.eye(3)
    t = row[1:4].copy()
    C = t if translation_mode == "C=t" else -R_wc @ t
    return R_wc, C

def H_gt(sequence, index_from, index_to, distance=None, quat_mode="qcw",
         translation_mode="C=t", use_rotation=True, normal=None):
    """H mapping ảnh index_from -> ảnh index_to (pixel). distance = khoảng cách camera0->mặt phẳng."""
    if sequence not in pose_data:
        raise ValueError(f"Không có pose data cho {sequence}")
    info = pose_data[sequence]
    K = info["K"]
    n_w = plane_normal(info["plane"]) if normal is None else normal
    R_i, C_i = pose(sequence, index_to, quat_mode, translation_mode, use_rotation)
    R_j, C_j = pose(sequence, index_from, quat_mode, translation_mode, use_rotation)
    _, C_ref = pose(sequence, 0, quat_mode, translation_mode, use_rotation)
    p_w = C_ref - distance * n_w
    d_j = float(n_w @ (p_w - C_j))
    if abs(d_j) < 1e-6:
        raise ValueError("Camera nằm trên mặt phẳng")
    H = K @ (R_i @ R_j.T + np.outer(R_i @ (C_j - C_i), R_j @ n_w) / d_j) @ np.linalg.inv(K)
    if not np.isfinite(H).all() or abs(H[2, 2]) < 1e-9:
        raise ValueError("H suy biến hoặc không hữu hạn")
    return H / H[2, 2]

def estimate_pair(file_a, file_b, ransac_px=3.0, ratio=0.75, max_features=4000):
    """Ghép SIFT ở độ phân giải gốc -> (H b->a, inliers, matches). Bộ nhớ giải phóng ngay sau khi đọc."""
    gray_a = cv2.imread(str(file_a), cv2.IMREAD_GRAYSCALE)
    gray_b = cv2.imread(str(file_b), cv2.IMREAD_GRAYSCALE)
    if gray_a is None or gray_b is None:
        raise ValueError("Không đọc được ảnh")
    sift = cv2.SIFT_create(nfeatures=max_features)
    kp_a, des_a = sift.detectAndCompute(gray_a, None)
    kp_b, des_b = sift.detectAndCompute(gray_b, None)
    del gray_a, gray_b
    if des_a is None or des_b is None or len(kp_a) < 4 or len(kp_b) < 4:
        raise ValueError("Không đủ đặc trưng SIFT")
    good = [m for pair in cv2.BFMatcher(cv2.NORM_L2).knnMatch(des_a, des_b, k=2)
            if len(pair) == 2 for m, n in [pair] if m.distance < ratio * n.distance]
    if len(good) < 4:
        raise ValueError(f"Chỉ có {len(good)} match tốt (< 4)")
    pa = np.float32([kp_a[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    pb = np.float32([kp_b[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    cv2.setRNGSeed(0)
    H, mask = cv2.findHomography(pb, pa, cv2.RANSAC, ransac_px)
    if H is None or mask is None:
        raise ValueError("RANSAC thất bại")
    return H / H[2, 2], int(mask.sum()), len(good)

def residual_px(H_est, H_gt_, size):
    W, H = size
    pts = np.float32([[0, 0], [W, 0], [W, H], [0, H], [W / 2, H / 2]]).reshape(-1, 1, 2)
    a = cv2.perspectiveTransform(pts, H_est).reshape(-1, 2)
    b = cv2.perspectiveTransform(pts, H_gt_).reshape(-1, 2)
    if not (np.isfinite(a).all() and np.isfinite(b).all()):
        raise ValueError("Phép chiếu không hữu hạn")
    return float(np.median(np.linalg.norm(a - b, axis=1)))

print("Đã nạp các hàm chuyển đổi hình học và chẩn đoán pose.")


### 3. Quét tìm quy ước toạ độ khớp nhất với ảnh thực tế
File dữ liệu gốc không ghi rõ thứ tự trục toạ độ và chiều quay của cảm biến. Chúng ta viết code quét qua các tổ hợp khả dĩ để tìm ra quy ước có sai số lệch so với ảnh thật là nhỏ nhất.

*Giải thích kết quả:* Sai số lệch (residual) không bao giờ bằng 0 tuyệt đối vì 3 lý do:
1. Địa hình mặt đất thực tế có gò đất, mương nước chứ không phẳng như tờ giấy.
2. Thiết bị đo GPS/IMU trên drone luôn có độ nhiễu nhất định.
3. Bản thân phép ghép ảnh SIFT cũng có sai số khoảng 1 pixel.
Do đó, tiêu chuẩn chọn là tìm tổ hợp có residual nhỏ nhất và ổn định nhất.


In [ ]:
best_row = None
rows = []
best_rot, best_norot = [], []
if not ENABLE_POSE_DIAGNOSIS or "phantom3-village-kfs" not in pose_data:
    print("Chẩn đoán pose không được kích hoạt hoặc không có pose data cho village.")
else:
    sequence = "phantom3-village-kfs"
    info = pose_data[sequence]
    H_ref, inliers, matches = estimate_pair(info["files"][0], info["files"][1])
    print(f"Tham chiếu SIFT (ảnh gốc): {matches} match, {inliers} inlier\n")

    def scan_distance(seq, i_from, i_to, quat_mode, translation_mode, use_rotation, normal, lo=20, hi=600, step=2.0):
        best = (np.inf, None, None)
        for distance in np.arange(lo, hi, step):
            for signed in (distance, -distance):
                try:
                    H = H_gt(seq, i_from, i_to, signed, quat_mode, translation_mode, use_rotation, normal)
                except ValueError:
                    continue
                value = residual_px(H_ref, H, info["size"])
                if value < best[0]:
                    best = (value, signed, H)
        return best

    for quat_mode, translation_mode, use_rotation in itertools.product(
            ("qcw", "qwc"), ("C=t", "minusRwTt"), (True, False)):
        best = scan_distance(sequence, 1, 0, quat_mode, translation_mode, use_rotation, plane_normal(info["plane"]))
        rows.append(dict(quat=quat_mode, translation=translation_mode, rotation=use_rotation,
                         best_px=best[0], distance=best[1]))
    best_row = min(rows, key=lambda r: r["best_px"]) if rows else None
    best_rot = [r for r in rows if r["quat"] == "qcw" and r["translation"] == "C=t" and r["rotation"]]
    best_norot = [r for r in rows if r["quat"] == "qcw" and r["translation"] == "C=t" and not r["rotation"]]
    if best_row:
        print(f"Tổ hợp khớp nhất: q={best_row['quat']}, tâm={best_row['translation']}, residual={best_row['best_px']:.2f} px")


### 4. Kiểm tra tỷ lệ chuyển động giữa Toạ độ bay và Khung hình ảnh
Nếu quy ước toạ độ là đúng, khi drone bay dịch chuyển một đoạn thực tế thì trên ảnh cũng phải dịch chuyển một khoảng pixel tương ứng theo đúng tỷ lệ quang học. Tỷ lệ này phải xấp xỉ bằng 1,0 trên chuỗi kiểm tra.


In [ ]:
def centre_shift_px(H, size):
    c = np.float32([[size[0] / 2, size[1] / 2]]).reshape(-1, 1, 2)
    return float(np.linalg.norm(cv2.perspectiveTransform(c, H).ravel() - c.ravel()))

medians = {}
fitted = None
if ENABLE_POSE_DIAGNOSIS and "phantom3-village-kfs" in pose_data:
    sequence = "phantom3-village-kfs"
    info = pose_data[sequence]
    distances = np.arange(120, 320, 2.0)
    ratios = {d: [] for d in distances}
    pairs = [(f, f + 1) for f in (0, 3, 6, 9, 12, 15)]
    for i_from, i_to in pairs:
        try:
            H_est, _, _ = estimate_pair(info["files"][i_from], info["files"][i_to])
        except ValueError:
            continue
        measured = centre_shift_px(H_est, info["size"])
        delta = np.linalg.norm(info["traj"][i_from][1:3] - info["traj"][i_to][1:3])
        for d in distances:
            predicted = delta * info["K"][0, 0] / d
            ratios[d].append(measured / predicted if predicted > 1e-9 else np.nan)
    medians = {d: float(np.nanmedian(v)) for d, v in ratios.items() if len(v) > 0 and np.isfinite(np.nanmedian(v))}
    fitted = min(medians, key=lambda d: abs(medians[d] - 1.0)) if medians else None
    if fitted is not None:
        print(f"Khoảng cách làm tỷ lệ ≈ 1: {fitted:.0f} m (tỷ lệ {medians[fitted]:.3f})")
else:
    print("Chẩn đoán pose không hoạt động; không tính fitted ratio.")


### 5. Đánh giá sai số tăng dần theo khoảng cách bay (Baseline)
Nếu dữ liệu toạ độ chuẩn xác tuyệt đối, khoảng cách giữa 2 ảnh càng xa thì sai số vẫn phải giữ nguyên. Thực tế cho thấy sai số tăng dần theo độ dài khoảng cách bay, chứng minh toạ độ drone bị tích luỹ sai lệch hệ thống.


In [ ]:
growth = {}
GROUND = None
if ENABLE_POSE_DIAGNOSIS and "phantom3-village-kfs" in pose_data and fitted is not None:
    GROUND = fitted
    sequence = "phantom3-village-kfs"
    info = pose_data[sequence]
    print(f"Dùng khoảng cách mặt phẳng = {GROUND:.0f} m\n")
    for gap in (1, 2, 4, 8, 12, 19):
        values = []
        for start in (0, 5, 10):
            try:
                H_est, _, _ = estimate_pair(info["files"][start], info["files"][start + gap])
                values.append(residual_px(H_est, H_gt(sequence, start + gap, start, GROUND), info["size"]))
            except ValueError:
                values.append(np.nan)
        valid_vals = [v for v in values if np.isfinite(v)]
        growth[gap] = float(np.median(valid_vals)) if valid_vals else None
    print("Growth theo gap:", growth)
else:
    print("Chẩn đoán pose không hoạt động; không tính growth theo baseline.")


In [ ]:
if ENABLE_POSE_DIAGNOSIS and "phantom3-village-kfs" in pose_data and medians and growth and fitted is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    info = pose_data["phantom3-village-kfs"]
    xs = sorted(medians)
    axes[0].plot(xs, [medians[d] for d in xs], "-", lw=2, color="tab:blue")
    axes[0].axhline(1.0, ls="--", color="k", lw=1)
    axes[0].axvline(fitted, ls=":", color="tab:green", lw=2)
    axes[0].set_title("Pose và ảnh cùng tỷ lệ")
    gaps = [g for g in sorted(growth) if growth[g] is not None]
    axes[1].plot(gaps, [growth[g] for g in gaps], "o-", lw=2, color="tab:orange")
    axes[1].set_title("Sai số đối chiếu pose theo gap")
    plt.tight_layout()
    plt.savefig(RUN_DIR / "trajectory_diagnosis.png", dpi=150)
    plt.show()
else:
    print("Bỏ qua vẽ đồ thị chẩn đoán pose (không kích hoạt hoặc không đủ dữ liệu).")


### 6. Kiểm tra tính tương thích trên chuỗi dữ liệu bãi cỏ (Grass Sequence)
Thực nghiệm cho thấy chuỗi dữ liệu bãi cỏ không dùng chung hệ toạ độ với chuỗi làng quê và bị lỗi suy biến hình học nghiêm trọng (toạ độ tâm camera đặt tại gốc 0, 0, 0).


In [ ]:
stats = {}
if ENABLE_POSE_DIAGNOSIS:
    for seq in SEQUENCES:
        if seq not in pose_data:
            continue
        info = pose_data[seq]
        _, C0 = pose(seq, 0)
        z0 = float(np.array([0.0, 0.0, 1.0]) @ (np.zeros(3) - C0))
        n_w = plane_normal(info["plane"])
        d_cfg = float(n_w @ (np.array(info["plane"][:3]) - C0))
        steps = np.linalg.norm(np.diff(info["traj"][:25, 1:4], axis=0), axis=1)
        stats[seq] = dict(t_origin=C0.tolist(), distance_z0=z0, distance_config_plane=d_cfg,
                          step_min=float(steps.min()), step_max=float(steps.max()))
        print(f"{seq}: tâm C0={C0.round(1).tolist()} | khoảng cách z0={z0:.1f} m")
else:
    print("Chẩn đoán pose tắt; không tính pose stats.")


### 7. Kết luận quan trọng của nhóm nghiên cứu
- Không thể dùng file `trajectory.txt` làm chuẩn đo drift chung cho toàn bộ dự án vì chất lượng dữ liệu không đồng đều giữa các cảnh bay.
- Nhóm thống nhất chuyển sang phương pháp đo **Chain vs Direct** ở Mục 8, hoàn toàn tự động, khách quan và không phụ thuộc vào thiết bị đo của drone (Quyết định `RDR-0007`).
- Kết quả chẩn đoán này được xuất ra file `trajectory_check.json` để làm bằng chứng khoa học.


In [ ]:
# Xuất báo cáo chẩn đoán pose (nếu bật) hoặc structured unavailable (nếu tắt/thiếu)
if ENABLE_POSE_DIAGNOSIS and best_row is not None:
    report_pose = {
        "status": "completed",
        "scope": "chẩn đoán cầu nối trajectory ↔ homography; KHÔNG chứa số liệu drift",
        "run_dir": str(RUN_DIR),
        "best_convention": best_row,
        "ground_distance_m": fitted,
        "residual_growth_px": growth,
        "sequence_stats": stats,
    }
else:
    report_pose = {
        "status": "disabled_or_unavailable",
        "scope": "chẩn đoán cầu nối trajectory ↔ homography không được kích hoạt hoặc không khả dụng",
        "run_dir": str(RUN_DIR),
        "best_convention": None,
        "ground_distance_m": None,
        "residual_growth_px": {},
        "sequence_stats": {},
    }
target_pose = RUN_DIR / "trajectory_check.json"
target_pose.write_text(json.dumps(report_pose, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Đã lưu kết quả chẩn đoán pose ({report_pose['status']}) vào: {target_pose}")


### 8. Phương pháp đo Drift bằng Chain vs Direct (RDR-0007)

### Nguyên lý đo tự nhất quán (Self-Consistent Drift):
- **Chain (Đi đường vòng):** Tích luỹ chuỗi homography từng bước: ảnh 0 sang 1, 1 sang 2... cho tới ảnh $i$. Kết quả đưa về hệ pixel ảnh gốc qua phép đổi hệ toạ độ $D \cdot H^{\text{chain}} \cdot D^{-1}$.
- **Direct (Đi đường thẳng):** Ghép trực tiếp ảnh $i$ về ảnh 0 bằng thuật toán tham chiếu chuẩn (SIFT 4000 keypoints, RANSAC 3 px ở ảnh gốc) tạo ma trận $H^{\text{direct}}$.
- **Đo sai số vòng:** Chiếu 5 điểm mốc (4 góc và tâm ảnh gốc) qua hai ma trận và tính sai số trung bình Euclidean pixel:
  $$\text{drift}_i = \frac{1}{|P|}\sum_{p \in P} \lVert H^{\text{chain}}_i p - H^{\text{direct}}_i p \rVert_2$$

### Quy tắc nghiệm thu khoa học theo RDR-0007 & RDR-0011:
1. **Mốc tham chiếu:** Tại $i = 0$, cả hai phép chiếu là identity (tự so với chính nó), sai số bằng 0. Điểm này là mốc toạ độ, không tính là một phép đo thực nghiệm độc lập.
2. **Ngưỡng hợp lệ:** Chỉ công nhận điểm đo tại $i > 0$ khi cặp ghép trực tiếp có **ít nhất 100 inliers** (inliers $\ge$ 100). Khi máy bay bay xa dần và mất vùng chồng hình ảnh với frame 0, kết quả được ghi nhận là không khả dụng (N/A) một cách trung thực, tuyệt đối không giả lập ground-truth.
3. **So sánh công bằng:** So sánh giữa 7 mô hình được thực hiện trên **tập khung hình đối chiếu chung (common eligible frames)** nơi tất cả các mô hình được so sánh đều có số liệu hợp lệ.


In [ ]:
def validate_benchmark_manifest(run_dir: Path, images_data: dict, manifest: dict):
    """Kiểm tra và bác bỏ nếu thứ tự frame hoặc hash ảnh không khớp manifest của benchmark."""
    manifest_inputs = manifest.get("inputs", [])
    if not manifest_inputs:
        raise ValueError(f"manifest.json tại {run_dir} không chứa danh sách 'inputs'")

    verified_inputs = {}
    for seq in SEQUENCES:
        seq_inputs = [inp for inp in manifest_inputs if seq in inp["path"]]
        if not seq_inputs:
            raise ValueError(f"Benchmark manifest không có dữ liệu input cho {seq}")

        files = images_data[seq]["files"]
        num_manifest_frames = len(seq_inputs)
        if len(files) < num_manifest_frames:
            raise ValueError(
                f"{seq}: Số file hiện tại ({len(files)}) ít hơn số frame trong benchmark manifest ({num_manifest_frames})"
            )

        for idx, inp in enumerate(seq_inputs):
            disk_file = files[idx]
            if disk_file.name != Path(inp["path"]).name:
                raise ValueError(
                    f"{seq} frame {idx}: Tên file không khớp thứ tự benchmark manifest! "
                    f"Hiện tại: {disk_file.name} vs Manifest: {Path(inp['path']).name}"
                )
            file_hash = hashlib.sha256(disk_file.read_bytes()).hexdigest()
            if file_hash != inp["sha256"]:
                raise ValueError(
                    f"{seq} frame {idx} ({disk_file.name}): Hash SHA256 không khớp benchmark manifest! "
                    f"Hiện tại: {file_hash} vs Manifest: {inp['sha256']}"
                )

        verified_inputs[seq] = {
            "num_frames": num_manifest_frames,
            "processed_hw": seq_inputs[0].get("processed_hw"),
        }
        print(f"PASS: {seq} đã kiểm chứng {num_manifest_frames} frame: thứ tự và SHA256 khớp 100% benchmark manifest.")

    return verified_inputs

verified_manifest_inputs = validate_benchmark_manifest(run_dir, images_data, benchmark_manifest)

def chain_to_original(run_dir, sequence, method, manifest, images_data):
    """
    H_chain (frame i -> frame 0) đưa về hệ pixel ẢNH GỐC.
    Sử dụng kích thước ảnh gốc từ images_data trực tiếp, không phụ thuộc pose metadata.
    """
    metrics_path = run_dir / "mosaic" / sequence / method / "metrics.json"
    if not metrics_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy metrics.json: {metrics_path}")
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    raw_mosaic = payload.get("H_to_mosaic", [])
    if not raw_mosaic:
        raise ValueError(f"H_to_mosaic rỗng: {sequence} / {method}")
    to_mosaic = [np.array(m, dtype=np.float64) for m in raw_mosaic]
    chain = [np.linalg.inv(to_mosaic[0]) @ m for m in to_mosaic]

    hw = [e["processed_hw"] for e in manifest.get("inputs", []) if sequence in e["path"]]
    if not hw or not hw[0]:
        raise ValueError(f"Không thấy kích thước xử lý cho {sequence} trong manifest")
    resized_h, resized_w = hw[0]
    original_w, original_h = images_data[sequence]["size"]
    D = np.diag([original_w / resized_w, original_h / resized_h, 1.0])
    D_inv = np.linalg.inv(D)
    return [D @ H @ D_inv for H in chain], payload.get("frames_stitched", len(chain)), payload.get("stopped_at_pair")

MIN_INLIERS = 100  # Ngưỡng tối thiểu theo RDR-0007 để công nhận điểm tham chiếu chuẩn

_sift_detector = None
_frame0_cache = {}

def get_sift_detector():
    global _sift_detector
    if _sift_detector is None:
        _sift_detector = cv2.SIFT_create(nfeatures=4000)
    return _sift_detector

def get_frame0_features(sequence, file0_path):
    if sequence not in _frame0_cache:
        gray0 = cv2.imread(str(file0_path), cv2.IMREAD_GRAYSCALE)
        if gray0 is None:
            raise ValueError(f"Không đọc được ảnh mốc frame 0: {file0_path}")
        sift = get_sift_detector()
        kp0, des0 = sift.detectAndCompute(gray0, None)
        _frame0_cache[sequence] = (kp0, des0)
    return _frame0_cache[sequence]

def direct_to_first(sequence, i):
    """
    H ghép thẳng frame i -> frame 0 bằng SIFT ảnh gốc.
    Trả về (H | None, inliers, matches, lý do).
    - i = 0: trả về identity, ghi rõ là reference_origin (không tính là điểm đo thực nghiệm).
    - i > 0: chỉ trả về H hợp lệ nếu inliers >= MIN_INLIERS (100).
    - Bắt đúng lỗi ngoại lệ I/O hoặc cv2 dự kiến, KHÔNG bắt bare Exception để tránh giấu lỗi code/OOM.
    """
    files = images_data[sequence]["files"]
    if i == 0:
        return np.eye(3), None, None, "reference_origin (identity)"
    if i >= len(files):
        return None, None, None, f"frame {i} vượt quá số ảnh có sẵn ({len(files)})"

    try:
        file0 = files[0]
        file_i = files[i]
        kp0, des0 = get_frame0_features(sequence, file0)
        if des0 is None or len(kp0) < 4:
            return None, 0, 0, "frame 0 không đủ đặc trưng SIFT"

        gray_i = cv2.imread(str(file_i), cv2.IMREAD_GRAYSCALE)
        if gray_i is None:
            return None, None, None, f"không đọc được ảnh frame {i}"

        sift = get_sift_detector()
        kp_i, des_i = sift.detectAndCompute(gray_i, None)
        del gray_i

        if des_i is None or len(kp_i) < 4:
            return None, 0, 0, "frame i không đủ đặc trưng SIFT"

        matches_pair = cv2.BFMatcher(cv2.NORM_L2).knnMatch(des0, des_i, k=2)
        good = [m for pair in matches_pair if len(pair) == 2 for m, n in [pair] if m.distance < 0.75 * n.distance]
        num_good = len(good)
        if num_good < 4:
            return None, 0, num_good, f"không đủ match tốt ({num_good} < 4)"

        p0 = np.float32([kp0[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        pi = np.float32([kp_i[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

        cv2.setRNGSeed(0)
        H, mask = cv2.findHomography(pi, p0, cv2.RANSAC, 3.0)
        if H is None or mask is None:
            return None, 0, num_good, "RANSAC thất bại"

        inliers = int(mask.sum())
        if inliers < MIN_INLIERS:
            return None, inliers, num_good, f"inliers {inliers} < {MIN_INLIERS} (không đủ chuẩn RDR-0007 / thiếu vùng chồng)"

        if not np.isfinite(H).all() or abs(H[2, 2]) < 1e-9:
            return None, inliers, num_good, "H suy biến hoặc không hữu hạn"

        return H / H[2, 2], inliers, num_good, "ok"
    except (cv2.error, OSError, ValueError) as exc:
        return None, None, None, f"lỗi kiểm tra ghép trực tiếp: {exc}"

def drift_points(size):
    W, H = size
    return np.float32([[0, 0], [W, 0], [W, H], [0, H], [W / 2, H / 2]]).reshape(-1, 1, 2)

def drift_px(H_chain, H_direct, size):
    """Sai số vòng: khoảng cách trung bình giữa hai phép chiếu trên 5 điểm mốc."""
    pts = drift_points(size)
    a = cv2.perspectiveTransform(pts, H_chain).reshape(-1, 2)
    b = cv2.perspectiveTransform(pts, H_direct).reshape(-1, 2)
    if not (np.isfinite(a).all() and np.isfinite(b).all()):
        raise ValueError("Phép chiếu không hữu hạn")
    return float(np.mean(np.linalg.norm(a - b, axis=1)))


### 9. Thực hiện tính toán Drift cho 7 mô hình trên cả 2 tập dữ liệu
Đọc chuỗi ma trận ghép ảnh của 7 mô hình đã lưu từ thực nghiệm benchmark và tính toán độ trôi sai số tích luỹ qua từng khung hình.


In [ ]:
# --- UNIT SELF-CHECKS (kiểm thử hành vi hình học của các helper thực tế trước khi chạy tính toán lớn) ---
def run_cloud_self_checks():
    test_size = (1920, 1080)
    H_ident = np.eye(3, dtype=np.float64)
    H_shift = np.array([[1.0, 0.0, 3.0], [0.0, 1.0, 4.0], [0.0, 0.0, 1.0]], dtype=np.float64)

    # 1. Kiểm tra drift_px với tịnh tiến (3, 4) -> khoảng cách Euclidean đúng bằng 5.0 px
    d_measured = drift_px(H_ident, H_shift, test_size)
    assert abs(d_measured - 5.0) < 1e-4, f"Self-check drift_px thất bại: {d_measured} != 5.0"

    # 2. Kiểm tra quat_to_R với identity quaternion
    R_ident = quat_to_R(np.array([0.0, 0.0, 0.0, 1.0]))
    assert np.allclose(R_ident, np.eye(3)), "Self-check quat_to_R thất bại"

    # 3. Kiểm tra centre_shift_px với tịnh tiến (3, 4) -> 5.0 px
    s_measured = centre_shift_px(H_shift, test_size)
    assert abs(s_measured - 5.0) < 1e-4, f"Self-check centre_shift_px thất bại: {s_measured} != 5.0"

    print("PASS: Tất cả các unit self-checks kiểm tra hành vi toán học helper đều thành công.")

run_cloud_self_checks()

# Hỗ trợ toàn bộ chuỗi (mặc định None) hoặc giới hạn qua biến môi trường UAV_MAX_DRIFT_FRAMES
max_frames_env = os.environ.get("UAV_MAX_DRIFT_FRAMES")
MAX_FRAMES = int(max_frames_env) if (max_frames_env and max_frames_env.isdigit()) else None
if MAX_FRAMES is not None:
    print(f"Cấu hình giới hạn khung hình đánh giá: MAX_FRAMES = {MAX_FRAMES}")
else:
    print("Cấu hình đánh giá toàn bộ khung hình có sẵn từ benchmark run (Full sequence mode).")

reference_cache = {}
results = []
sequence_chains = {}
sequence_coverage = {}

# Bước 1: Trích xuất chuỗi H_chain và tính toán drift từng bước
for sequence in SEQUENCES:
    size = images_data[sequence]["size"]
    files = images_data[sequence]["files"]
    sequence_chains[sequence] = {}
    sequence_coverage[sequence] = {}

    for method in METHODS:
        chain, stitched_count, stopped_at_pair = chain_to_original(
            run_dir, sequence, method, benchmark_manifest, images_data
        )
        sequence_chains[sequence][method] = chain
        sequence_coverage[sequence][method] = {
            "frames_stitched": stitched_count,
            "stopped_at_pair": stopped_at_pair,
        }

        n = min(len(chain), MAX_FRAMES) if MAX_FRAMES is not None else len(chain)
        series = []
        inlier_log = []

        for i in range(n):
            key = (sequence, i)
            if key not in reference_cache:
                reference_cache[key] = direct_to_first(sequence, i)
            H_direct, inliers, matches, reason = reference_cache[key]

            frame_entry = dict(
                i=i,
                frame_name=files[i].name if i < len(files) else f"frame_{i}",
                inliers=inliers,
                matches=matches,
                status=reason,
                is_reference_origin=(i == 0)
            )
            inlier_log.append(frame_entry)

            if i == 0:
                # Frame 0 là mốc identity (đo tự so với chính nó = 0)
                series.append(0.0)
                continue

            if H_direct is None:
                series.append(None)
                continue

            # Đo độ lệch tương đối giữa phép ghép tuần tự và phép ghép trực tiếp SIFT
            val = drift_px(chain[i], H_direct, size)
            series.append(val)

        # Lọc các điểm đo thực nghiệm (i >= 1) có tham chiếu hợp lệ
        measured_experimental = [(i, v) for i, v in enumerate(series) if i > 0 and v is not None]
        last_i = measured_experimental[-1][0] if measured_experimental else None
        last_v = measured_experimental[-1][1] if measured_experimental else None

        method_ate = float(np.sqrt(np.mean([v ** 2 for _, v in measured_experimental]))) if measured_experimental else None
        skipped = [e["i"] for e, v in zip(inlier_log, series) if v is None and e["i"] > 0]

        record = dict(
            sequence=sequence,
            method=method,
            frames_requested=len(chain),
            frames_stitched=stitched_count,
            stopped_at_pair=stopped_at_pair,
            evaluated_frames=n,
            measured_steps=len(measured_experimental),
            last_index=last_i,
            final_drift_px=last_v,
            method_ate_px=method_ate,
            skipped_steps=skipped,
            series=series,
            inliers=inlier_log,  # BẢO LƯU TOÀN BỘ LOG INLIER VÀ FRAME IDS
        )

        ckpt_path = RUN_DIR / "checkpoints" / f"{sequence}_{method}.json"
        ckpt_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
        results.append(record)

    print(f"Xong tính toán drift chuỗi cho {sequence}")

# Bước 2: Xác định Common Eligible Frame Set cho từng sequence để so sánh công bằng cross-method
common_eligible_sets = {}
for sequence in SEQUENCES:
    seq_records = [r for r in results if r["sequence"] == sequence]
    max_steps = max(len(r["series"]) for r in seq_records)
    common_indices = []
    for i in range(1, max_steps):
        ref_ok = (reference_cache.get((sequence, i)) is not None and reference_cache[(sequence, i)][0] is not None)
        if not ref_ok:
            continue
        all_methods_valid = all(i < len(r["series"]) and r["series"][i] is not None for r in seq_records)
        if all_methods_valid:
            common_indices.append(i)
    common_eligible_sets[sequence] = common_indices
    print(f"Sequence {sequence}: Tập khung hình đối chiếu chung (common eligible frames i>=1): {common_indices}")

    for r in seq_records:
        if common_indices:
            common_vals = [r["series"][idx] for idx in common_indices if r["series"][idx] is not None]
            r["common_ate_px"] = float(np.sqrt(np.mean([v ** 2 for v in common_vals]))) if common_vals else None
            r["common_eligible_count"] = len(common_indices)
        else:
            r["common_ate_px"] = None
            r["common_eligible_count"] = 0

# Bước 3: In bảng tổng hợp kết quả phân tách rõ ràng coverage và drift trên common eligible frames
header = f"{'sequence':20} {'method':20} {'stitched':>8} {'common steps':>12} {'common ATE (px)':>16} {'last idx':>10} {'final drift':>12}"
print(header)
print("-" * len(header))
for r in sorted(results, key=lambda r: (r["sequence"], r["common_ate_px"] if r["common_ate_px"] is not None else 99999)):
    comm_ate_str = f"{r['common_ate_px']:16.2f}" if r["common_ate_px"] is not None else f"{'N/A':>16}"
    final_dr_str = f"{r['final_drift_px']:12.2f}" if r["final_drift_px"] is not None else f"{'N/A':>12}"
    print(f"{r['sequence']:20} {r['method']:20} {r['frames_stitched']:8} {r['common_eligible_count']:12} "
          f"{comm_ate_str} {str(r['last_index']):>10} {final_dr_str}")


In [ ]:
fig, axes = plt.subplots(1, len(SEQUENCES), figsize=(8 * len(SEQUENCES), 5))
for ax, sequence in zip(np.ravel(axes), SEQUENCES):
    for r in [x for x in results if x["sequence"] == sequence]:
        xs = [i for i, v in enumerate(r["series"]) if v is not None]
        ys = [v for v in r["series"] if v is not None]
        if not xs:
            continue
        style = dict(lw=2.5, color="k", ls="--") if r["method"] == "sift_bf" else dict(lw=1.5)
        label = r["method"] + (" (SIFT chain vs SIFT direct)" if r["method"] == "sift_bf" else "")
        ax.plot(xs, ys, marker="o", ms=3, label=label, **style)
    ax.set_title(f"Drift tích luỹ (chain vs direct) - {sequence}")
    ax.set_xlabel("khung hình trong chuỗi (i)")
    ax.set_ylabel("sai số vòng (px, ảnh gốc)")
    ax.legend(fontsize=7); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(RUN_DIR / "drift_curves.png", dpi=150)
plt.show()


### 10. Đánh giá kết quả và Kiểm tra tính toàn vẹn cấu trúc
Hệ thống kiểm tra tự động:
- Khung hình đầu tiên ($i = 0$) bắt buộc có sai số vòng bằng 0 (dung sai số thực $< 10^{-4}$).
- Mọi điểm đo được đưa vào phân tích ($i > 0$) bắt buộc phải thoả mãn điều kiện nghiệm thu $\ge 100$ inlier của RDR-0007.
- Báo cáo được xuất kèm đầy đủ provenance (hash SHA256 của benchmark manifest, cấu hình môi trường) và bảo lưu toàn bộ nhật ký inlier.
- Ghi `completed.json` sau cùng khi tất cả các bước và artifact đều hoàn tất thành công.


In [ ]:
# 1. Kiểm tra cấu trúc: đủ kết quả cho các mô hình và sequence
assert len(results) == len(METHODS) * len(SEQUENCES), f"Thiếu kết quả: {len(results)} != {len(METHODS) * len(SEQUENCES)}"

# 2. Kiểm tra mốc: i = 0 là identity -> sai số vòng bắt buộc bằng 0 (với sai số dấu phẩy động < 1e-4)
for r in results:
    assert abs(r["series"][0] - 0.0) < 1e-4, f"Mốc i=0 sai khác 0: {r['sequence']} / {r['method']}: {r['series'][0]}"

# 3. Kiểm tra tính tuân thủ RDR-0007: mọi điểm đo thực nghiệm i > 0 có số liệu đều đạt >= MIN_INLIERS
for r in results:
    for entry, value in zip(r["inliers"], r["series"]):
        if value is not None and entry["i"] > 0:
            assert entry["inliers"] is not None and entry["inliers"] >= MIN_INLIERS, (
                f"Điểm đo không đủ điều kiện RDR-0007 lọt vào kết quả: {r['method']} step {entry['i']} inliers={entry['inliers']}"
            )

print("PASS: Các kiểm tra tính toàn vẹn cấu trúc và quy tắc nghiệm thu RDR-0007 đều đạt.")

# 4. Quan sát thực nghiệm (Ghi nhận khách quan, KHÔNG dùng assert ép kết quả nghiên cứu)
baseline_metrics = {}
for s in SEQUENCES:
    sift_res = next((r for r in results if r["sequence"] == s and r["method"] == "sift_bf"), None)
    if sift_res:
        baseline_metrics[s] = {
            "final_drift_px": sift_res["final_drift_px"],
            "common_ate_px": sift_res["common_ate_px"],
            "last_index": sift_res["last_index"],
        }
        print(f"Mức nền SIFT tham chiếu ({s}): ATE={sift_res['common_ate_px']} px | Last ({sift_res['last_index']})={sift_res['final_drift_px']} px")

# 5. Xuất báo cáo kết quả chi tiết với đầy đủ provenance và log inliers
report = {
    "scope": "drift tự nhất quán (chain vs direct) theo RDR-0007; đơn vị PIXEL ảnh gốc",
    "metric_decision": "docs/logs/decisions/RDR-0007_drift_metric_choice.md",
    "evaluation_protocol": "docs/logs/decisions/RDR-0011_loso_and_technical_readiness.md",
    "provenance": {
        "source_run_dir": str(run_dir),
        "source_manifest_path": str(run_dir / "manifest.json"),
        "source_manifest_sha256": manifest_hash,
        "run_dir": str(RUN_DIR),
        "environment": {
            "numpy": np.__version__,
            "opencv": cv2.__version__,
            "python": sys.version.split()[0],
            "platform": platform.platform(),
        },
    },
    "reference_matcher": {
        "detector": "SIFT (OpenCV)",
        "nfeatures": 4000,
        "ratio": 0.75,
        "ransac_px": 3.0,
        "min_inliers_gate": MIN_INLIERS,
        "resolution_by_sequence": {s: list(images_data[s]["size"]) for s in SEQUENCES},
    },
    "marker_points": "4 góc + tâm ảnh gốc",
    "direction": "H_chain và H_direct đều ánh xạ từ frame i về frame 0",
    "baseline_sift_reference": baseline_metrics,
    "common_eligible_sets": common_eligible_sets,
    "caveats_and_limitations": [
        "thước đo đo độ bất đồng tự nhất quán (self-consistency) giữa ghép tuần tự và ghép trực tiếp SIFT, không phải ground-truth tuyệt đối trên thực địa",
        "mức nền phương pháp phụ thuộc vào sai số của matcher tham chiếu SIFT",
        "chỉ đo được khi hai frame có đủ vùng chồng hình ảnh (ngưỡng 100 inlier RDR-0007); mất vùng chồng khi bay xa được ghi nhận là không khả dụng (N/A) một cách trung thực, không giả lập ground-truth",
        "so sánh giữa các mô hình phải dựa trên tập khung hình đối chiếu chung (common eligible frames) để đảm bảo tính công bằng",
    ],
    # BẢO LƯU ĐẦY ĐỦ INLIER LOG, FRAME IDS VÀ METADATA THEO YÊU CẦU AUDIT
    "results": results,
}

drift_target = RUN_DIR / "drift_selfconsistent.json"
drift_target.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
print("Đã lưu báo cáo drift vào:", drift_target)

# 6. GHI MARKER HOÀN TẤT CUỐI CÙNG (completed.json LAST after success)
completed_marker = {
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "status": "success",
    "experiment": "04_drift_evaluation",
    "run_dir": str(RUN_DIR),
    "source_run_dir": str(run_dir),
    "source_manifest_sha256": manifest_hash,
    "sequences": list(SEQUENCES),
    "methods": list(METHODS),
    "artifacts": [
        "drift_selfconsistent.json",
        "drift_curves.png",
        *[name for name in ("trajectory_check.json", "trajectory_diagnosis.png") if (RUN_DIR / name).is_file()],
    ],
}
completed_path = RUN_DIR / "completed.json"
completed_path.write_text(json.dumps(completed_marker, indent=2), encoding="utf-8")
print("Đã ghi completed marker thành công:", completed_path)
